# Laboratorio: teorema espectral y diagonalización ortogonal

Construiremos bases ortonormales de vectores propios, proyectores espectrales y funciones de matrices simétricas y hermitianas.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=6, suppress=True)

## 1. Diagonalización unitaria exacta

La misma función sirve para matrices simétricas reales y hermitianas complejas. Ortonormaliza únicamente dentro de cada espacio propio y verifica $Q^*Q=I$ y $Q^*AQ=D$.

In [ ]:
def adjunta(M):
    M = sp.Matrix(M)
    return sp.conjugate(M).T

def diagonalizacion_unitaria(A):
    A = sp.Matrix(A)
    if A.rows != A.cols:
        raise ValueError("La matriz debe ser cuadrada.")
    if A != adjunta(A):
        return None, None, None

    columnas, valores, datos = [], [], []
    for lam, mult_alg, base in A.eigenvects():
        base_on = sp.GramSchmidt(base, orthonormal=True)
        datos.append((lam, mult_alg, base, base_on))
        columnas.extend(base_on)
        valores.extend([lam] * len(base_on))

    Q = sp.Matrix.hstack(*columnas)
    D = sp.diag(*valores)
    Qstar = adjunta(Q)
    assert sp.simplify(Qstar*Q) == sp.eye(A.rows)
    assert sp.simplify(Qstar*A*Q) == D
    assert sp.simplify(Q*D*Qstar) == A
    return Q, D, datos

## 2. Ejemplo real completo

Usamos la matriz simétrica de las evaluaciones del curso. El valor propio $2$ es repetido, de modo que su espacio propio necesita una base ortonormal.

In [ ]:
A = sp.Matrix([[6, 2, 2], [2, 3, 1], [2, 1, 3]])
Q, D, datos = diagonalizacion_unitaria(A)
print("¿A es simétrica?", A == A.T)
for lam, ma, base, base_on in datos:
    print(f"\nlambda={lam}, multiplicidad algebraica={ma}")
    print("Base inicial:", base)
    print("Base ortonormal:", base_on)
print("\nQ =")
sp.pprint(Q)
print("D =")
sp.pprint(D)
assert sorted(D.diagonal()) == [2, 2, 8]

### Gram–Schmidt se aplica dentro del espacio propio repetido

Mostramos que la base inicial de $E_2$ no tiene por qué ser ortogonal, mientras que la base procesada sí lo es.

In [ ]:
dato_2 = next(dato for dato in datos if dato[0] == 2)
base_2, base_2_on = dato_2[2], dato_2[3]
print("Producto interno de la base inicial:", (base_2[0].T*base_2[1])[0])
print("Producto interno después de Gram-Schmidt:",
      sp.simplify((base_2_on[0].T*base_2_on[1])[0]))
assert sp.simplify((base_2_on[0].T*base_2_on[1])[0]) == 0
assert all(A*v == 2*v for v in base_2_on)

## 3. Potencias y raíz cúbica

La diagonalización ortogonal permite calcular $A^6$ y una raíz cúbica simétrica actuando directamente sobre los valores propios.

In [ ]:
A6 = sp.simplify(Q * D**6 * Q.T)
A6_esperada = sp.Matrix([
    [174784, 87360, 87360],
    [87360, 43744, 43680],
    [87360, 43680, 43744],
])
assert A6 == A**6 == A6_esperada
print("A^6 =")
sp.pprint(A6)

D_raiz = sp.diag(*[sp.real_root(lam, 3) for lam in D.diagonal()])
C = sp.simplify(Q * D_raiz * Q.T)
assert sp.simplify(C**3 - A) == sp.zeros(3)
assert C == C.T

## 4. Proyectores espectrales ortogonales

Agrupamos las columnas de $Q$ según su valor propio. Cada suma $qq^T$ proyecta ortogonalmente sobre el espacio propio correspondiente.

In [ ]:
proyectores = {}
for j, lam in enumerate(D.diagonal()):
    q = Q.col(j)
    proyectores[lam] = proyectores.get(lam, sp.zeros(3)) + q*q.T

Pi2, Pi8 = map(sp.simplify, (proyectores[2], proyectores[8]))
print("Proyector sobre E_8:")
sp.pprint(Pi8)
print("Proyector sobre E_2:")
sp.pprint(Pi2)

assert Pi2.T == Pi2 and Pi8.T == Pi8
assert Pi2**2 == Pi2 and Pi8**2 == Pi8
assert Pi2*Pi8 == sp.zeros(3)
assert Pi2 + Pi8 == sp.eye(3)
assert 2*Pi2 + 8*Pi8 == A
assert sp.simplify(A - (2*sp.eye(3) + 6*Pi8)) == sp.zeros(3)

## 5. Cociente de Rayleigh

Para vectores unitarios, $x^TAx$ debe permanecer entre los valores propios extremos $2$ y $8$. Verificamos esta cota con una muestra reproducible.

In [ ]:
rng = np.random.default_rng(2026)
X = rng.normal(size=(5000, 3))
X /= np.linalg.norm(X, axis=1, keepdims=True)
A_np = np.array(A, dtype=float)
rayleigh = np.einsum('bi,ij,bj->b', X, A_np, X)
print("mínimo muestral:", rayleigh.min())
print("máximo muestral:", rayleigh.max())
assert rayleigh.min() >= 2 - 1e-12
assert rayleigh.max() <= 8 + 1e-12

### Visualización en dos dimensiones

Para $M=\begin{pmatrix}2&1\\1&2\end{pmatrix}$, evaluamos el cociente de Rayleigh sobre el círculo unitario. Sus extremos son los valores propios $1$ y $3$.

In [ ]:
M = np.array([[2., 1.], [1., 2.]])
theta = np.linspace(0, 2*np.pi, 800)
U = np.vstack([np.cos(theta), np.sin(theta)])
Rtheta = np.einsum('in,ij,jn->n', U, M, U)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(theta, Rtheta, color='#2a6fbb', lw=2)
ax.axhline(1, color='#1b9e77', ls='--', label=r'$\lambda_{\min}=1$')
ax.axhline(3, color='#d95f02', ls='--', label=r'$\lambda_{\max}=3$')
ax.set(xlabel=r'$\theta$', ylabel=r'$R_M((\cos\theta,\sin\theta))$',
       xlim=(0, 2*np.pi), xticks=[0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(['0', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
ax.grid(alpha=0.25)
ax.legend()
plt.show()
assert np.isclose(Rtheta.min(), 1, atol=2e-5)
assert np.isclose(Rtheta.max(), 3, atol=2e-5)

## 6. Caso hermitiano complejo

La matriz siguiente tiene entradas complejas, pero es hermitiana. Sus valores propios son reales y admite una diagonalización unitaria.

In [ ]:
H = sp.Matrix([[2, sp.I], [-sp.I, 2]])
U_H, D_H, datos_H = diagonalizacion_unitaria(H)
print("¿H es hermitiana?", H == adjunta(H))
print("Valores propios:", list(D_H.diagonal()))
sp.pprint(U_H)
assert set(D_H.diagonal()) == {1, 3}
assert sp.simplify(adjunta(U_H)*U_H) == sp.eye(2)
assert sp.simplify(U_H*D_H*adjunta(U_H)) == H

## 7. Diagonalizable no significa ortogonalmente diagonalizable

La matriz $N=\begin{pmatrix}7&1\\2&8\end{pmatrix}$ tiene dos valores propios distintos y es diagonalizable, pero no es simétrica. Por tanto, no puede escribirse como $QDQ^T$ con $Q$ ortogonal y $D$ diagonal real.

In [ ]:
N = sp.Matrix([[7, 1], [2, 8]])
Q_N, D_N, datos_N = diagonalizacion_unitaria(N)
v6 = sp.Matrix([-1, 1])
v9 = sp.Matrix([1, 2])
print("¿N es simétrica?", N == N.T)
print("Producto de sus direcciones propias:", (v6.T*v9)[0])
assert Q_N is None and D_N is None
assert (v6.T*v9)[0] != 0

## 8. Cálculo numérico con NumPy

Para matrices reales simétricas conviene usar **numpy.linalg.eigh**: aprovecha la simetría y devuelve vectores propios ortonormales.

In [ ]:
valores_np, Q_np = np.linalg.eigh(A_np)
D_np = np.diag(valores_np)
residuo = np.linalg.norm(A_np@Q_np - Q_np@D_np)
error_ortogonalidad = np.linalg.norm(Q_np.T@Q_np - np.eye(3))
print("Valores propios:", valores_np)
print("Residuo espectral:", residuo)
print("Error de ortogonalidad:", error_ortogonalidad)
assert np.allclose(valores_np, [2, 2, 8])
assert residuo < 1e-12 and error_ortogonalidad < 1e-12

## 9. Actividades

1. Cambia la base inicial de $E_2$ y comprueba que el proyector $\Pi_2$ no cambia.
2. Construye $A^{1/2}$ mediante los proyectores espectrales cuando todos los valores propios son no negativos.
3. Verifica que una matriz simétrica e idempotente produce proyectores espectrales asociados solo a $0$ y $1$.
4. Repite el experimento del cociente de Rayleigh con una matriz simétrica de orden $4$.
5. Compara **numpy.linalg.eig** y **numpy.linalg.eigh** sobre una matriz simétrica perturbada por un pequeño error no simétrico.

## 10. Cierre

- Los operadores autoadjuntos tienen espectro real y espacios propios distintos ortogonales.
- El teorema espectral garantiza una base ortonormal completa de vectores propios.
- En espacios propios repetidos, Gram–Schmidt permite elegir una base ortonormal.
- Los proyectores espectrales de un autoadjunto son proyecciones ortogonales.
- El cociente de Rayleigh queda acotado por los valores propios extremos.